# English to Dutch Transformer Model Training
This notebook covers the full workflow for English to Dutch translation on the opus_books dataset (about 38k sentence pairs). Run the sections in order, or skip training if you already have a saved model on Drive.


## Setup
Check the GPU and install the required packages.


In [ ]:
import torch, warnings; warnings.filterwarnings('ignore')
print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()} | device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
!nvidia-smi -L

In [ ]:
%%capture
!pip install -q datasets tokenizers torchmetrics tensorboard altair pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/transformers-from-scratch_weights
!mkdir -p /content/drive/MyDrive/transformers-from-scratch_runs


## Configuration
Set the language pair and where to save weights. Using Drive keeps checkpoints and tokenizers after the Colab runtime restarts.


In [ ]:
import pathlib, os
if not pathlib.Path("/content/transformers-from-scratch").exists():
    !git clone https://github.com/abhijitdalal26/transformers-from-scratch.git /content/transformers-from-scratch
%cd /content/transformers-from-scratch/pytorch-transformer
!ls -lh model.py dataset.py config.py train.py translate.py
!cat config.py


In [ ]:
from config import get_config, get_weights_file_path, latest_weights_file_path
cfg = get_config()
print("Original cfg:", cfg)
assert cfg['lang_src']=='en' and cfg['lang_tgt']=='nl', "should be en-nl (Dutch)"
cfg['model_folder'] = "/content/drive/MyDrive/transformers-from-scratch_weights"
cfg['tokenizer_file'] = "/content/drive/MyDrive/transformers-from-scratch_weights/tokenizer_{0}.json"
cfg['experiment_name'] = "/content/drive/MyDrive/transformers-from-scratch_runs/tmodel_en_nl"
cfg['num_epochs'] = 20  # set 3-5 for quick smoke test
cfg['batch_size'] = 8
cfg['preload'] = 'latest'  # if Drive already has tmodel_*.pt, resume from there
print("Effective cfg:", cfg)
print("Latest checkpoint on Drive:", latest_weights_file_path(cfg))
print("\n'model already available' = latest_weights_file_path(cfg) != None → train_model will set initial_epoch = saved_epoch+1 and skip 0..saved. If you open notebook just to demo, we can SKIP the 3.5h train cell entirely.")


## Train
Train the model. Full training is about 3.5 hours for 20 epochs on a T4 GPU. If a checkpoint already exists on Drive you can set SKIP_TRAIN to True and go straight to the next sections.


In [ ]:
SKIP_TRAIN = False  # set True to skip 3.5h and jump to Inference/Visuals if weights exist
import pathlib as _pl
ckpt = latest_weights_file_path(cfg)
if SKIP_TRAIN and ckpt is not None:
    print(f"Skipping train — using existing {ckpt}")
else:
    if ckpt is not None and "tmodel_19" in ckpt and not SKIP_TRAIN:
        print(f"Found full training {ckpt} — set SKIP_TRAIN=True next time to save 3.5h")
    try:
        import huggingface_hub.utils._hf_uris as _hf_uris
        _orig = _hf_uris._parse_repo_body
        def _patched(*args, **kwargs):
            try:
                return _orig(*args, **kwargs)
            except Exception:
                raw = None
                for v in list(args) + list(kwargs.values()):
                    if isinstance(v, str) and "hf://datasets/opus_books" in v:
                        raw = v
                        break
                if raw is not None:
                    raw2 = raw.replace("hf://datasets/opus_books", "hf://datasets/huggingface/opus_books")
                    new_args = tuple(raw2 if (isinstance(a, str) and a == raw) else a for a in args)
                    new_kwargs = {k: (raw2 if (isinstance(val, str) and val == raw) else val) for k, val in kwargs.items()}
                    return _orig(*new_args, **new_kwargs)
                raise
        _hf_uris._parse_repo_body = _patched
        print("Patched HF Hub URI parser for opus_books")
    except Exception as e:
        print(f"HF patch skipped: {e}")
    from train import train_model
    train_model(cfg)  # ~0.6h/epoch en-nl


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/transformers-from-scratch_runs --port 6006


## Inference
Try a few translations. Each sentence takes less than a second.


In [ ]:
from translate import translate
import config as cfgmod
orig = cfgmod.get_config; cfgmod.get_config = lambda: cfg  # make translate use Drive cfg
print(translate("Hello, how are you?"))
print("---")
print(translate("I love learning languages."))
print("---")
print(translate("The book is on the table."))
cfgmod.get_config = orig
print("\n# By index (shows SOURCE/TARGET/PREDICTED from val set):")
cfgmod.get_config = lambda: cfg; print(translate(42)); cfgmod.get_config = orig

## Inference — Step by Step
Follow one sentence end-to-end. Each option shows how the model moves from text to tokens to predictions.


### 1. Tokenization
See how text becomes tokens and ids.


In [ ]:
# Ensure model and tokenizers are ready — loads from Drive if not already in memory
import torch
try:
    _ = model and tokenizer_src and tokenizer_tgt and device
    print(f"Reusing loaded model on {device}")
except NameError:
    print("Loading model and tokenizers from Drive...")
    from train import get_ds, get_model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")
    _, _, tokenizer_src, tokenizer_tgt = get_ds(cfg)
    model = get_model(cfg, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
    ckpt = latest_weights_file_path(cfg)
    print(f"checkpoint: {ckpt}")
    if ckpt is not None:
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state['model_state_dict'])
        print(f"Loaded epoch {state['epoch']} global_step {state['global_step']}")
    else:
        print("No checkpoint found — using random weights for demo (train first for real translations)")
    model.eval()

# Pick any sentence — edit this line to try your own
sentence = "Hello, how are you?"  # try: "The book is on the table." or "I love learning languages."
print(f"\nInput sentence: {sentence}")
tokens = tokenizer_src.encode(sentence).tokens
ids = tokenizer_src.encode(sentence).ids
print("Tokens:", tokens)
print("Token ids:", ids)
print(f"Vocab sizes: src={tokenizer_src.get_vocab_size()} tgt={tokenizer_tgt.get_vocab_size()}")
print(f"Special ids: PAD={tokenizer_src.token_to_id('[PAD]')} SOS={tokenizer_src.token_to_id('[SOS]')} EOS={tokenizer_src.token_to_id('[EOS]')} UNK={tokenizer_src.token_to_id('[UNK]')}")


### 2. Padding and Mask
Add special tokens and pad to the model length. The mask tells the encoder which tokens are real.


In [ ]:
seq_len = cfg["seq_len"]
sos_id = tokenizer_src.token_to_id("[SOS]")
eos_id = tokenizer_src.token_to_id("[EOS]")
pad_id = tokenizer_src.token_to_id("[PAD]")
padded_ids = [sos_id] + ids + [eos_id] + [pad_id] * (seq_len - len(ids) - 2)
padded_tokens = ["[SOS]"] + tokens + ["[EOS]"] + ["[PAD]"] * (seq_len - len(tokens) - 2)
print(f"Padded length: {len(padded_ids)} (seq_len={seq_len})")
print("First 20 padded tokens:", padded_tokens[:20])
print("First 20 padded ids:", padded_ids[:20])

source = torch.tensor(padded_ids, dtype=torch.int64).unsqueeze(0).to(device)  # (1, seq_len)
source_mask = (source != pad_id).unsqueeze(0).unsqueeze(0).int().to(device)  # (1,1,1,seq_len)
print(f"\nSource tensor shape: {source.shape} mask shape: {source_mask.shape}")
print(f"Non-pad tokens: {source_mask.sum().item()} of {seq_len}")


### 3. Embeddings and Encoder
Tokens become vectors. Positional information is added inside the embedding, then the encoder builds a representation.


In [ ]:
src_emb = model.src_embed(source)  # token embeddings + positional encoding
print(f"Token embeddings shape: {src_emb.shape}  (batch, seq_len, d_model={cfg['d_model']})")
with torch.no_grad():
    encoder_output = model.encode(source, source_mask)
print(f"Encoder output shape: {encoder_output.shape}")
print(f"Encoder output sample (first token, first 8 dims): {encoder_output[0,0,:8].cpu().numpy().round(3)}")


### 4. Decoder — One Token at a Time
The decoder predicts the next token, adds it to the input, and repeats until it predicts the end token.


In [ ]:
import torch.nn.functional as F
from dataset import causal_mask

print(f"Generating translation step by step for: '{sentence}'\n")
sos_tgt = tokenizer_tgt.token_to_id("[SOS]")
eos_tgt = tokenizer_tgt.token_to_id("[EOS]")
decoder_input = torch.empty(1, 1).fill_(sos_tgt).type_as(source).to(device)  # start with SOS
print(f"Step 0: decoder_input = [SOS] id={sos_tgt} -> '{tokenizer_tgt.decode([sos_tgt])}'")

max_steps = 30  # limit for display; full seq_len is 350
for step in range(1, max_steps+1):
    decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)
    out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
    logits = model.project(out[:, -1])  # (1, vocab_tgt) — raw scores for last position
    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, 5, dim=-1)
    next_id = torch.argmax(logits, dim=-1).item()
    next_token = tokenizer_tgt.decode([next_id])
    
    prefix_ids = decoder_input[0].cpu().tolist()
    prefix_tokens = [tokenizer_tgt.decode([i]) for i in prefix_ids]
    
    print(f"\n--- Step {step} ---")
    print(f"  Decoder input so far: {prefix_tokens}  ids={prefix_ids}")
    print(f"  Logits shape: {logits.shape}  vocab={tokenizer_tgt.get_vocab_size()}  sample logits (first 5): {logits[0,:5].detach().cpu().numpy().round(2)}")
    print(f"  Top 5 candidates:")
    for rank in range(5):
        tid = top_ids[0, rank].item()
        tok = tokenizer_tgt.decode([tid])
        p = top_probs[0, rank].item()
        marker = " <- chosen" if tid == next_id else ""
        print(f"    {rank+1}. id={tid:4d} token='{tok}' prob={p:.4f}{marker}")
    print(f"  Chosen: id={next_id} token='{next_token}'  Full prefix now: '{tokenizer_tgt.decode(decoder_input[0].cpu().tolist() + [next_id])}'")
    
    decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(source).fill_(next_id).to(device)], dim=1)
    if next_id == eos_tgt:
        print(f"\nReached [EOS] at step {step} — translation complete.")
        break
else:
    print(f"\nStopped after {max_steps} steps (no EOS yet).")

final_text = tokenizer_tgt.decode(decoder_input[0].cpu().tolist())
print(f"\nFinal translation: {final_text}")


## Validation
Evaluate translation quality on the validation set.


In [ ]:
import torch
from train import get_ds, get_model, run_validation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(cfg)
model = get_model(cfg, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
ckpt = latest_weights_file_path(cfg)
print("ckpt", ckpt)
state = torch.load(ckpt, map_location=device)
model.load_state_dict(state['model_state_dict'])
print(f"Loaded epoch {state['epoch']} global_step {state['global_step']}")
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device, lambda m: print(m), state['global_step'], None, num_examples=5)

## Attention Visual
Heatmaps per layer and head. Twelve charts per type keep the view clear and fast. Expand the last cell if you want all eight heads.


### Helpers
Convert the attention matrix to a table for Altair.


In [ ]:
import altair as alt, pandas as pd, numpy as np
from train import greedy_decode

def mtx2df(m, max_row, max_col, row_tokens, col_tokens):
    return pd.DataFrame([ (r,c,float(m[r,c]), f"{r:03d} {row_tokens[r] if len(row_tokens)>r else '<blank>'}", f"{c:03d} {col_tokens[c] if len(col_tokens)>c else '<blank>'}") for r in range(m.shape[0]) for c in range(m.shape[1]) if r<max_row and c<max_col], columns=["row","column","value","row_token","col_token"])


### Map
Pick one attention matrix and turn it into a heatmap.


In [ ]:
def get_attn_map(attn_type: str, layer: int, head: int):
    if attn_type=="encoder": attn = model.encoder.layers[layer].self_attention_block.attention_scores
    elif attn_type=="decoder": attn = model.decoder.layers[layer].self_attention_block.attention_scores
    else: attn = model.decoder.layers[layer].cross_attention_block.attention_scores
    return attn[0, head].data

def attn_map(attn_type, layer, head, row_tokens, col_tokens, max_len):
    df = mtx2df(get_attn_map(attn_type, layer, head), max_len, max_len, row_tokens, col_tokens)
    return alt.Chart(df).mark_rect().encode(x=alt.X("col_token", axis=alt.Axis(title="")), y=alt.Y("row_token", axis=alt.Axis(title="")), color="value", tooltip=["row","column","value","row_token","col_token"]).properties(height=400,width=400,title=f"{attn_type} L{layer} H{head}").interactive()


### Grid
Combine maps for several layers and heads.


In [ ]:
def get_all_attention_maps(attn_type, layers, heads, row_tokens, col_tokens, max_len):
    charts=[]
    for layer in layers:
        rowCharts=[attn_map(attn_type, layer, head, row_tokens, col_tokens, max_len) for head in heads]
        charts.append(alt.hconcat(*rowCharts))
    return alt.vconcat(*charts)

def load_next_batch():
    batch = next(iter(val_dataloader))
    encoder_input = batch["encoder_input"].to(device)
    encoder_mask = batch["encoder_mask"].to(device)
    decoder_input = batch["decoder_input"].to(device)
    decoder_mask = batch["decoder_mask"].to(device)
    enc_tokens = [tokenizer_src.id_to_token(idx) for idx in encoder_input[0].cpu().numpy()]
    dec_tokens = [tokenizer_tgt.id_to_token(idx) for idx in decoder_input[0].cpu().numpy()]
    assert encoder_input.size(0)==1
    model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
    return batch, enc_tokens, dec_tokens


### Load Example
Run the model once to fill the attention scores, then show an example sentence.


In [ ]:
batch, enc_tokens, dec_tokens = load_next_batch()
print(f"SOURCE: {batch['src_text'][0]}")
print(f"TARGET: {batch['tgt_text'][0]}")
sent_len = enc_tokens.index("[PAD]") if "[PAD]" in enc_tokens else 20
print(f"sent_len {sent_len} trunc 20")


### Encoder Self-Attention
How each source word looks at other source words.


In [ ]:
layers=[0,1,2]; heads=[0,1,2,3]  # 12 charts — use [0,1,2,3,4,5,6,7] for all eight
get_all_attention_maps("encoder", layers, heads, enc_tokens, enc_tokens, min(20, sent_len))

### Decoder Self-Attention
How each target word looks at previous target words (causal).


In [ ]:
get_all_attention_maps("decoder", [0,1,2], [0,1,2,3], dec_tokens, dec_tokens, min(20, sent_len))

### Encoder-Decoder Attention
How each target word looks at the source words.


In [ ]:
get_all_attention_maps("encoder-decoder", [0,1,2], [0,1,2,3], enc_tokens, dec_tokens, min(20, sent_len))

### More Heads (Optional)
Uncomment to see all eight heads — heavier but complete.


In [ ]:
# layers=[0,1,2]; heads=[0,1,2,3,4,5,6,7]
# get_all_attention_maps("encoder", layers, heads, enc_tokens, enc_tokens, min(20, sent_len))

## Beam Search
Keep several candidates instead of just the best one at each step. Often improves quality.


### Define Beam Search


In [ ]:
from dataset import causal_mask
def beam_search_decode(model, beam_size, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id('[SOS]'); eos_idx = tokenizer_tgt.token_to_id('[EOS]')
    encoder_output = model.encode(source, source_mask)
    decoder_initial = torch.empty(1,1).fill_(sos_idx).type_as(source).to(device)
    candidates = [(decoder_initial, 0.0)]  # (seq, log_score)
    while True:
        if any(c.size(1)==max_len for c,_ in candidates): break
        new_cands=[]
        for cand, score in candidates:
            if cand[0][-1].item()==eos_idx: new_cands.append((cand, score)); continue
            cand_mask = causal_mask(cand.size(1)).type_as(source_mask).to(device)
            out = model.decode(encoder_output, source_mask, cand, cand_mask)
            prob = model.project(out[:,-1])  # (1, vocab)
            log_prob = torch.log_softmax(prob, dim=1)
            topk_log, topk_idx = torch.topk(log_prob, beam_size, dim=1)
            for i in range(beam_size):
                token = topk_idx[0][i].unsqueeze(0).unsqueeze(0)
                new_cand = torch.cat([cand, token], dim=1)
                new_score = score + topk_log[0][i].item()
                new_cands.append((new_cand, new_score))
        candidates = sorted(new_cands, key=lambda x: x[1], reverse=True)[:beam_size]
        if all(c[0][-1].item()==eos_idx for c,_ in candidates): break
    return sorted(candidates, key=lambda x: x[1], reverse=True)[0][0].squeeze(0)  # best


### Compare with Greedy


In [ ]:
# Demo on a validation batch
batch = next(iter(val_dataloader))
src = batch["encoder_input"].to(device); src_mask=batch["encoder_mask"].to(device)
print("SOURCE:", batch["src_text"][0])
print("TARGET:", batch["tgt_text"][0])
greedy = greedy_decode(model, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("GREEDY  :", tokenizer_tgt.decode(greedy.cpu().numpy()))
beam4 = beam_search_decode(model, 4, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("BEAM(4) :", tokenizer_tgt.decode(beam4.cpu().numpy()))
beam8 = beam_search_decode(model, 8, src, src_mask, tokenizer_src, tokenizer_tgt, cfg['seq_len'], device)
print("BEAM(8) :", tokenizer_tgt.decode(beam8.cpu().numpy()))


### Notes
If a checkpoint is found on Drive, training resumes from that epoch automatically. Set SKIP_TRAIN to True when you only want to run inference or visualization without retraining.
